# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² dataset using the `mlcroissant` library, with all Croissant entities referenced by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure latest `mlcroissant` and pandas are installed
!pip install --upgrade mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL (as provided)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Dataset identifier (@id): {metadata.id}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), fields, and their `@id`s. All entities are referenced using their `@id` fields.

In [ ]:
# List available record sets, their @id and basic metadata
print("Available RecordSets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}, description: {getattr(rs, 'description', 'No description')}")

# For each record set, list field @id, name, and data type
for rs in record_sets:
    print(f"\nFields in RecordSet '@id': {rs.id}: {rs.name}")
    for field in rs.fields:
        print(f"  - Field @id: {field.id}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for analysis. Reference record sets and fields by their `@id` as listed above.

In [ ]:
# Load records for all record sets into DataFrames, using the record set @id for each
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs in record_sets:
    print(f"Loading records for RecordSet '@id': {rs.id}")
    records = list(dataset.records(record_set=rs.id))
    if len(records):
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Columns: {list(df.columns)}\nSample:")
        display(df.head())
    else:
        print("(No records found)")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records based on numeric fields, normalization, and grouping. Reference all columns/fields by their `@id`.

In [ ]:
### Select a record set and examine its numeric fields by @id ###
import numpy as np

# For demonstration, pick the first non-empty record set
for rs_id, df in dataframes.items():
    # Identify numeric fields
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        print(f"\nWorking with RecordSet: {rs_id}")
        numeric_field = numeric_fields[0]  # choose the first numeric field by @id
        print(f"Using numeric field: {numeric_field}")
        break

# Set a filter threshold
threshold = df[numeric_field].quantile(0.9)  # Use 90th percentile as an example filter
# Filter records where this numeric field exceeds the threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (top 10%):")
display(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Select a group/categorical field by @id if one exists
categorical_fields = df.select_dtypes(include=[object]).columns.tolist()
if categorical_fields:
    group_field = categorical_fields[0]
    print(f"\nGrouping by field (by @id): {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
    display(grouped_df.head())
else:
    print("No suitable categorical fields for grouping found.")

## 5. Visualization
Visualize the distribution or relationship between fields. All field references use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (by @id)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If a groupable categorical field exists, visualize mean values per group
if categorical_fields:
    # Show means per group (if groupable)
    means_by_group = df.groupby(group_field)[numeric_field].mean().nlargest(12)
    plt.figure(figsize=(10,5))
    means_by_group.plot(kind='bar', color='skyblue')
    plt.title(f"Mean {numeric_field} grouped by {group_field} (by @id)")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to explore and process the FAIR² dataset using the `mlcroissant` library. 
- All Croissant entities (record sets, fields/columns) are referenced using their `@id` in code and comments.
- Data are loaded, selected, and grouped in a reproducible manner.
- This structure can be extended for deeper analysis or other Croissant-compliant datasets.